# Knowledge Base Retrieval Evaluation

This notebook evaluates the performance of the knowledge base retrieval system against manual human annotations.

### Evaluation Setup
- **Sampled Questions**: 30 unique questions randomly sampled from `data/processed/test.csv` (`random_state=42`).
- **Candidate Pool**: Top-10 chunks retrieved for each question using FAISS cosine similarity.
- **Blinding**: Chunks were presented in shuffled order without ranks or scores for unbiased human labeling in `results/retrieval_labeling.csv`.
- **Metrics Computed**:
  - Precision@1, Precision@3, Precision@5
  - Recall@3, Recall@5 (*measured against the labeled pool of top-10 chunks*)
  - Mean Reciprocal Rank (MRR)
- **Baseline**: Random baseline retrieving k chunks uniformly at random from the knowledge base (1000 repetitions, seed 42).

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set project root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from config import RETRIEVAL_MODEL_NAME
from src.retrieval_metrics import (
    compute_retrieval_metrics,
    compute_random_baseline,
    compare_retrieval_to_baseline,
    find_retrieval_failures,
    get_real_retrieval_ranking,
)

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results', exist_ok=True)


C:\Users\Kartik Patel\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Human Annotations
We load `results/retrieval_labeling.csv` containing candidate chunks.

In [2]:
labeling_path = Path('../results/retrieval_labeling.csv')
if not labeling_path.exists():
    raise FileNotFoundError(f'Labeling file not found at {labeling_path}. Run scripts/make_retrieval_labels.py first.')

df_labels = pd.read_csv(labeling_path)
print(f'Loaded {len(df_labels)} labeling rows ({df_labels["question_id"].nunique()} unique questions).')
df_labels.head(10)


Loaded 300 labeling rows (30 unique questions).


,question_id,question_text,chunk_id,chunk_text,relevant
0,1.1,What is the role of a prototype program in pro...,trees_and_traversals_000,"# Trees, Hierarchical Data Structures, and Tra...",NaN
1,1.1,What is the role of a prototype program in pro...,functions_and_recursion_001,"optimization passes like constant folding, tho...",NaN
2,1.1,What is the role of a prototype program in pro...,classes_and_oop_000,# Object-Oriented Programming and Class Design...,NaN
3,1.1,What is the role of a prototype program in pro...,pointers_and_memory_000,"# Pointers, Memory Addresses, and Function Poi...",NaN
4,1.1,What is the role of a prototype program in pro...,classes_and_oop_001,as soon as a programmer supplies any customize...,NaN
5,1.1,What is the role of a prototype program in pro...,stacks_000,# Stacks: Abstract Data Type and Practical App...,NaN
6,1.1,What is the role of a prototype program in pro...,functions_and_recursion_000,"# Functions, Signatures, Overloading, and Recu...",NaN
7,1.1,What is the role of a prototype program in pro...,pointers_and_memory_001,"referenced content), and constant pointer to c...",NaN
8,1.1,What is the role of a prototype program in pro...,cxx_basics_and_problem_solving_000,# Software Engineering and Problem Solving in ...,NaN
9,1.1,What is the role of a prototype program in pro...,cxx_basics_and_problem_solving_001,establishes regression test suites that dictat...,NaN


## 2. Check Annotation Completion Status
Check if the `relevant` column has been filled with manual labels (1 or 0).

In [3]:
non_empty = df_labels['relevant'].dropna()
valid_labels = non_empty[non_empty.astype(str).str.strip() != '']
is_labeled = len(valid_labels) > 0

print(f'Total annotations available: {len(valid_labels)} / {len(df_labels)}')
if not is_labeled:
    print('\n[INFO] Manual labeling is in progress in results/retrieval_labeling.csv.')
    print('Once labeled, rerun this notebook to compute evaluation metrics and save results/retrieval_results.csv.')
else:
    print('Relevance distribution:')
    print(valid_labels.value_counts())


Total annotations available: 0 / 300

[INFO] Manual labeling is in progress in results/retrieval_labeling.csv.
Once labeled, rerun this notebook to compute evaluation metrics and save results/retrieval_results.csv.


## 3. Retrieval Performance vs Random Baseline
Compute Precision@1, @3, @5, Recall@3, @5, and MRR. Recall is measured against the labeled pool (top-10).
A random baseline retrieves chunks uniformly at random over 1000 repetitions (seed 42).

In [4]:
if is_labeled:
    comparison_df = compare_retrieval_to_baseline(
        labeled_df=df_labels,
        n_repetitions=1000,
        seed=42,
        chunks_path='../data/kb/chunks.csv',
        index_path='../data/kb/index.faiss',
        model_name=RETRIEVAL_MODEL_NAME,
    )
    # Save to results/retrieval_results.csv
    results_path = Path('../results/retrieval_results.csv')
    comparison_df.to_csv(results_path, index=False)
    print(f'Saved retrieval evaluation results to {results_path}\n')
    display(comparison_df)
else:
    print('Waiting for manual labels. Displaying template results table:')
    template_df = pd.DataFrame([
        {'Method': f'Retriever ({RETRIEVAL_MODEL_NAME})', 'Precision@1': np.nan, 'Precision@3': np.nan, 'Precision@5': np.nan, 'Recall@3': np.nan, 'Recall@5': np.nan, 'MRR': np.nan},
        {'Method': 'Random Baseline (1000 runs)', 'Precision@1': np.nan, 'Precision@3': np.nan, 'Precision@5': np.nan, 'Recall@3': np.nan, 'Recall@5': np.nan, 'MRR': np.nan},
    ])
    display(template_df)


Waiting for manual labels. Displaying template results table:


,Method,Precision@1,Precision@3,Precision@5,Recall@3,Recall@5,MRR
0,Retriever (sentence-transformers/all-MiniLM-L6...,NaN,NaN,NaN,NaN,NaN,NaN
1,Random Baseline (1000 runs),NaN,NaN,NaN,NaN,NaN,NaN


## 4. Failure Analysis: Questions with No Relevant Chunk in Top-3
Identifies questions where the retriever failed to surface any relevant chunk in the top 3 results.

In [5]:
if is_labeled:
    failures_df = find_retrieval_failures(
        labeled_df=df_labels,
        k=3,
        chunks_path='../data/kb/chunks.csv',
        index_path='../data/kb/index.faiss',
        model_name=RETRIEVAL_MODEL_NAME,
    )
    print(f'Total Retrieval Failures in Top-3: {len(failures_df)} / {df_labels["question_id"].nunique()} questions')
    if len(failures_df) > 0:
        display(failures_df)
    else:
        print('No retrieval failures! Top-3 retrieved at least one relevant chunk for every question.')
else:
    print('Failure analysis will be evaluated once labeling is complete.')


Failure analysis will be evaluated once labeling is complete.


## 5. Visualizing Comparison
Bar chart comparing retriever metrics against the random baseline.

In [6]:
if is_labeled:
    metric_cols = ['Precision@1', 'Precision@3', 'Precision@5', 'Recall@3', 'Recall@5', 'MRR']
    plot_df = comparison_df.melt(id_vars=['Method'], value_vars=metric_cols, var_name='Metric', value_name='Score')

    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x='Metric', y='Score', hue='Method', palette='Blues_r')
    plt.title(f'Knowledge Base Retrieval Evaluation: {RETRIEVAL_MODEL_NAME} vs Random Baseline')
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    fig_path = '../results/figures/retrieval_eval_comparison.png'
    plt.savefig(fig_path, dpi=300)
    print(f'Figure saved to {fig_path}')
    plt.show()
